# Web Scraping
v.ekc

Not all data comes with an API. Today we learn to fetch web pages programmatically and parse their HTML structure to extract tables and content using BeautifulSoup.

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | Scraping a Basic Webpage |
| 3 | Scraping a Complex Page......Cal Poly Humboldt |
| 4 | Building a DataFrame from Scraped Tables |
| 5 | 🔬 Activity: Humboldt Geographic Data |
| Appendix | Quick Reference |

---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Whenever you want to scrape a website without an API
import io
import requests
from bs4 import BeautifulSoup

---
## 2. Scraping a Basic Webpage

We'll start with a very simple page to learn the basics of HTML parsing. BeautifulSoup documentation [here](https://www.crummy.com/software/BeautifulSoup/bs4/doc/).

Let's look at [this](https://web.ics.purdue.edu/~gchopra/class/public/pages/webdesign/05_simple.html) very simple webpage.

###

| Code | What it does |
|------|--------------|
| `requests.get(url)` | Fetch a webpage |
| `site.status_code` | `200` = success |
| `BeautifulSoup(site.text)` | Parse raw HTML |
| `soup.find('tag')` | First matching element |
| `soup.find_all('tag')` | All matching elements (returns list) |
| `element['attr']` | Get an attribute (e.g., `element['href']`) |
| `element.text` | Get the text content of an element |

We're starting with an intentionally bare-bones webpage without JavaScript or dynamic content. This lets us focus on the HTML parsing mechanics before dealing with a real site's complexity. Once these basics click, the same tools work on any page.

In [ ]:
# Get the content of a website
site = requests.get('https://web.ics.purdue.edu/~gchopra/class/public/pages/webdesign/05_simple.html')

In [ ]:
# What did we just get?
type(site)

In [ ]:
# Check the status
site.status_code

In [ ]:
# With APIs, the output was typically JSON, but not regular webpages
site.json() # error!

Web APIs return structured JSON. Regular webpages return **raw HTML** which appears as a long string of tags, attributes, and content all tangled together. `.text` gives us that raw string. The next step is teaching Python to make sense of it.

In [ ]:
# inspect the contents with text
site.text
# messy!

That raw HTML is technically readable but practically unusable and you'd have to write your own string parsing to find anything. **BeautifulSoup** parses that string into a tree of objects you can navigate with simple method calls. Think of it as turning a wall of text into a searchable document.

In [ ]:
# Let's ****beautify**** this and make it easier to parse
soup = BeautifulSoup(site.text)

In [ ]:
# What does our soup look like?
soup

In [87]:
soup.prettify()

'<html>\n <head>\n  <title>\n   A very simple webpage\n  </title>\n  <basefont size="4"/>\n </head>\n <body bgcolor="FFFFFF">\n  <h1>\n   A very simple webpage. This is an "h1" level header.\n  </h1>\n  <h2>\n   This is a level h2 header.\n  </h2>\n  <h6>\n   This is a level h6 header.  Pretty small!\n  </h6>\n  <p>\n   This is a standard paragraph.\n  </p>\n  <p align="center">\n   Now I\'ve aligned it in the center of the screen.\n  </p>\n  <p align="right">\n   Now aligned to the right\n  </p>\n  <p>\n   <b>\n    Bold text\n   </b>\n  </p>\n  <p>\n   <strong>\n    Strongly emphasized text\n   </strong>\n   Can you tell the difference vs. bold?\n  </p>\n  <p>\n   <i>\n    Italics\n   </i>\n  </p>\n  <p>\n   <em>\n    Emphasized text\n   </em>\n   Just like Italics!\n  </p>\n  <p>\n   Here is a pretty picture:\n   <img alt="Pretty Picture" src="example/prettypicture.jpg"/>\n  </p>\n  <p>\n   Same thing, aligned differently to the paragraph:\n   <img align="top" alt="Pretty Picture" sr

In [ ]:
# Make it even prettier
print(soup.prettify())

Now we can start pulling out specific pieces. Two key methods:
- **`soup.find('tag')`** — returns the *first* matching element, or `None` if not found
- **`soup.find_all('tag')`** — returns a *list* of all matching elements (empty list if none)

HTML elements also carry **attributes** (like `href` on links) and **text content** (the visible words on screen). We access these separately.

### Parse the html

In [ ]:
# Find a level 1 header
soup.find('h1')

In [ ]:
# Find a level 2 header
soup.find('h2')

In [ ]:
# Find all the level 2 headers
soup.find_all('h2')

In [ ]:
# Find all the level 3 headers
soup.find_all('h3')
# There were none.

In [ ]:
# Find all the paragraphs
soup.find_all('p')

In [ ]:
# Find all the hyperlinks
soup.find_all('a')

An `<a>` tag has two useful pieces: the **URL** (stored in the `href` attribute) and the **display text** (what you see on screen). These are accessed differently:
- `element['href']` — gets an attribute value, like a dictionary key
- `element.text` — gets all the text nested inside the tag

In [ ]:
# Get the links
soup.find('a')['href']

In [ ]:
# Get the text for the hyperlink
soup.find('a').text

HTML has two list types: **`<ol>`** (ordered / numbered) and **`<ul>`** (unordered / bulleted). Both contain **`<li>`** items. `find_all('li')` returns items from both — use `find_all('ol')` or `find_all('ul')` when you need to distinguish them.

In [ ]:
# Get all the list items
soup.find_all('li')

In [ ]:
# Specifically get the ordered lists
soup.find_all('ol')

In [ ]:
# Specifically get the unordered lists
soup.find_all('ul')

---
### Try It 1: Parse the Simple Page 😊

Use the `soup` object from the simple Purdue page above.

1. Use `find_all('p')` to get all paragraph elements. How many are there? Extract the text from each one into a Python list using a list comprehension.

2. Find all hyperlinks (`<a>` tags). For each one, print both the display text and the URL it points to.

3. The page has both ordered and unordered lists. How many total `<li>` items are there? Can you tell which items belong to the `<ol>` vs the `<ul>`?

In [ ]:
# 1. All paragraphs → list of text



In [ ]:
# 2. All links → print text + href



In [ ]:
# 3. Count li items; split by ol vs ul



<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

```python
# 1.
paragraphs = [paragraphs.text for paragraphs in soup.find_all('p')]
print(len(paragraphs), paragraphs)

# 2.
for a in soup.find_all('a'):
    print(a.text, '"', a['href'])

# 3.
print('Total li:', len(soup.find_all('li')))
for ol in soup.find_all('ol'):
    print('ordered list items:', [li.text for li in ol.find_all('li')])
for ul in soup.find_all('ul'):
    print('unordered list items:', [li.text for li in ul.find_all('li')])
```

</details>

---
## 3. Scraping a Complex Page: Cal Poly Humboldt

Let's check out [this](https://www.humboldt.edu/irar/fall-semester-fast-facts) Cal Poly Humboldt website and extract a specific data table.

### 📋 Board Reference

| Code | What it does |
|------|--------------|
| `soup.find_all('tag', class_='cls')` | Filter by CSS class |
| `soup.select('tag.cls')` | CSS selector shorthand |
| `soup.find_all('table')` | All HTML tables |
| `table.find_all('tr')` | All rows in a table |
| `row.find_all('td')` | All cells in a row |

In [ ]:
# Get the data
cph_stats = requests.get('https://www.humboldt.edu/irar/fall-semester-fast-facts')
cph_stats.status_code

In [ ]:
# Beautify the data
cph_soup = BeautifulSoup(cph_stats.text, 'html.parser')

In [ ]:
# Pretty!
print(cph_soup.prettify())

In [ ]:
# check for specific tags
cph_soup.find_all('a')

In [ ]:
# refine the search with css selectors
cph_soup.find_all('a', class_ = "top-level")

In [ ]:
# A shorthand way of searching that
cph_soup.select('a.top-level')

In [ ]:
# How many tables are there?
len(cph_soup.find_all('table'))

There are multiple tables on the page. The trick to finding the right one is to look at nearby HTML landmarks — headings, IDs, or class names that label each section. Here we check the `<h3>` headers to pair them up with the table index we want.

In [ ]:
# Tables are labeled with h3 headers
cph_soup.find_all('h3')

In [ ]:
# Let's just focus on one of the tables
#  just look at the website to select the 3rd indexed table

student_ethnicity_table = cph_soup.find_all('table')[3]
student_ethnicity_table


# another method if you are looking for the h3 that points to a table
#     you search the h3 for matching term
#     then we use .find_next('table') to return the next table element

#target_h3 = cph_soup.find('h3', string=lambda t: t and 'Ethnicity' in t)
#target_table = target_h3.find_next('table')


An HTML table is nested three levels deep:
- **`<table>`** — the whole table
- **`<tr>`** — a table row
- **`<td>`** — an individual data cell (`<th>` for header cells)

To reach a single value, drill down: table → rows → cells → `.text`.

In [ ]:
# Look at the rows
student_ethnicity_table.find_all('tr')

In [ ]:
# Look at a single row
student_ethnicity_table.find_all('tr')[0]

In [ ]:
# Look at all the data points in that row
student_ethnicity_table.find_all('tr')[0].find_all('td')

In [ ]:
# Look at one specific data point
student_ethnicity_table.find_all('tr')[0].find_all('td')[1]

In [ ]:
# Get the text of that data point 
student_ethnicity_table.find_all('tr')[0].find_all('td')[1].text

---
### Try It 2: Navigate a Table on the CPH Page 😊

Use the `cph_soup` object from the Cal Poly Humboldt page.

1. Print the `<h3>` text that appears just before each table on the page, so you know what each table contains.

2. Pick any table that is **not** `student_ethnicity_table`. Extract the text from its first data row as a Python list.

3. Use `cph_soup.select()` with a CSS selector to find all `<a>` elements with class `'top-level'`. How many are there, and what text do they display?

In [ ]:
# 1. Print h3 label paired with each table



In [ ]:
# 2. First data row from a different table



In [ ]:
# 3. CSS selector for a.top-level


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

```python
# 1.
headers = cph_soup.find_all('h3')
tables  = cph_soup.find_all('table')
for h, t in zip(headers, tables):
    print(h.text.strip())

# 2.
other_table = cph_soup.find_all('table')[0]
first_row   = other_table.find_all('tr')[0]
[cell.text.strip() for cell in first_row.find_all('td')]

# 3.
links = cph_soup.select('a.top-level')
print(len(links))
[a.text for a in links]
```

</details>

---
## 4. Building a DataFrame from Scraped Tables

Two approaches: a manual loop, or the `pd.read_html()` shortcut.

### Manual approach — nested loop

The goal is to convert the HTML table into a list of lists — one inner list per row, one string per cell — then hand that to `pd.DataFrame()`. The outer loop walks rows (`<tr>`), the inner loop walks cells (`<td>`) within each row.

In [ ]:
# Create a nested list with the data
table_vals = []

for i in student_ethnicity_table.find_all('tr'):
    row_i = []
    for j in i.find_all('td'):
        row_i.append(j.text)
    table_vals.append(row_i)

In [ ]:
# Check out the result
table_vals

The raw result has two things to fix:
- **Row 0 is actually the header** — HTML header cells (`<th>`) were treated the same as data cells (`<td>`), so the column labels landed in row 0 as data. We need to promote them.
- **`\xa0`** — this is HTML's non-breaking space (`&nbsp;`). It appears wherever the page used `&nbsp;` for formatting and becomes a literal `\xa0` character when scraped. Watch for it as a phantom index label or cell value.

In [ ]:
# Make it a dataframe
df = pd.DataFrame(table_vals)
df


# notice cell [0,0] is emptpy...not even a ``
#   there is an invisible \ax0 there?

In [ ]:
# Clean it up (reset column labels)
df.columns = df.iloc[0]
df.drop(0,inplace=True)
df

In [ ]:
# weird \axa0

df.columns[0]

In [ ]:
# Clean it up (reset row labels)
# make sure to get rid of the \axa0


df.set_index('\xa0',inplace=True)


In [ ]:
df

In [ ]:
# Would require further cleanup
df.dtypes

Writing a nested loop every time you want a table is tedious. Pandas has a built-in shortcut: **`pd.read_html()`** — give it an HTML string, and it finds every `<table>` element and returns a list of DataFrames automatically. The one catch: it expects a *string*, not a BeautifulSoup object, so we wrap with `str()` first.

### Pandas shortcut — pd.read_html()

In [ ]:
# We've taken a chunk of html that we want to parse
student_ethnicity_table

In [ ]:
# What is the type
type(student_ethnicity_table)

In [ ]:
# Read the html into a dataframe
pd.read_html(io.StringIO(str(student_ethnicity_table)))[0]

In [ ]:
# We can use it on the whole page if we want to
pd.read_html(io.StringIO(cph_stats.text))

---
### Try It 3: Build and Clean a Scraped DataFrame 😊

Use `student_ethnicity_table` and the `df` DataFrame built with the manual loop.

1. The `df` index is named `\xa0` which is a non-breaking space artifact from the HTML. Rename the index to `'Ethnicity'` so the DataFrame is properly labeled.

2. The count columns were scraped as strings. Pick one column and convert it to a numeric dtype. What character in the values might cause `int()` to fail, and how do you handle it? How would you change the `Percentage` to a `float` type?

3. Run `pd.read_html(io.StringIO(str(student_ethnicity_table)))[0]` and compare it to your manually-cleaned `df`. Does `pd.read_html()` handle the `\xa0` and column headers automatically, or do you still need to clean it?

In [ ]:
# 1. Rename the \xa0 index to 'Ethnicity'




In [ ]:
# 2a. Convert a count column from string to numeric




# 2b. Convert `Percentage` to `float`



In [ ]:
# 3. Compare pd.read_html() output to manual approach



<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*`\xa0` is the HTML non-breaking space. Commas in numbers like `'1,234'` will break `int()` — strip them first with `.str.replace()`.*

```python
# 1.
df.index.name = 'Ethnicity'

# 2.
df['Percentage'] = df['Percentage'].str.replace('%', '').astype(float)

# 3.
clean = pd.read_html(str(student_ethnicity_table))[0]
clean
# pd.read_html() usually handles column headers automatically
# but may still carry \xa0 in cell values — check clean.dtypes
```

</details>

---
## 5. 🔬 Activity — Humboldt Geographic Data

---
### Scrape & Clean a Table

1. Identify the table **Fall 2025 Geographic Origin of Current Students** from the soup object that we created called `cph_soup`.

2. Pull out the data from the table using a loop. Then, convert your list into a `DataFrame`.

4. Clean the DataFrame: fix column names, fix the index name, fix dtypes, and remove any artifacts from the HTML structure.

5. **Lastly:** Use `pd.read_html()` as a shortcut instead. Does it give a cleaner result?

In [ ]:
# 1. Identify the  the Geographic Origin table

target2 = cph_soup.find('h3', string=lambda t: t and 'Geographic Origin' in t)
target_table2 = target2.find_next('table')


In [ ]:
# 2. Now turn scrape out the  a table using a loop

df2=[]
for i in target_table2.find_all('tr'):
    row = []
    for j in i.find_all('td'):
        row.append(j.text)
    df2.append(row)
df2 = pd.DataFrame(df2)

df2

In [ ]:
# 3. Clean the DataFrame

# rename the columns

df2.rename(columns = {0:'Region', 1: 'Undergraduate', 2:'Post Baccalaureate', 3:'Totals', 4: 'Percentage'}, inplace=True)
df2


# possibly clean up that first row!
df2.drop(0, inplace=True)


# make the index -> Region
df2.set_index('Region')

In [ ]:
# 3. Clean the DataFrame cont

# check out the dataypes
#   change if necessary....(it is!)




In [ ]:
# 4. Try pd.read_html() shortcut


---
## Appendix — Web Scraping Quick Reference

### Minimal template
```python
import requests
from bs4 import BeautifulSoup

site = requests.get('https://example.com')
soup = BeautifulSoup(site.text, 'html.parser')

soup.find('h1')            # first h1 element
soup.find_all('p')         # all paragraphs
soup.find('a')['href']     # link URL
soup.find('a').text        # link text
```

### CSS selectors
```python
soup.find_all('a', class_='nav')   # by class
soup.select('a.nav')               # CSS selector shorthand
```

### Table → DataFrame
```python
# Manual loop
table = soup.find_all('table')[0]
rows = [[cell.text for cell in row.find_all('td')]
        for row in table.find_all('tr')]
df = pd.DataFrame(rows)

# Shortcut
pd.read_html(str(table))[0]
pd.read_html(site.text)     # all tables on the page
```